# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [16]:
import google.generativeai as genai
print("Import successful!")

Import successful!


/tmp/ipykernel_1071/1299244211.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [3]:
import google.generativeai as genai
print("Library imported successfully!")


Library imported successfully!


/tmp/ipykernel_1272/3668684794.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [11]:

# @title User input needed.

# Documents for the ReAct agent to search through.
DOCUMENTS = {
    "space_missions": (
        "NASA's Apollo 11 mission landed the first humans on the Moon in July 1969. "
        "Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited the Moon. "
        "The mission used the Saturn V rocket to travel to space. "
        "Apollo 11 returned safely to Earth on July 24, 1969."
    ),
    "mars_rovers": (
        "NASA's Perseverance rover landed on Mars in February 2021. "
        "Its mission includes searching for signs of ancient microbial life. "
        "The rover collects and stores rock samples for possible future return to Earth. "
        "It also carries an experiment called MOXIE that demonstrated oxygen production from the Martian atmosphere."
    ),
    "space_telescope": (
        "The James Webb Space Telescope launched in December 2021. "
        "It observes the universe primarily in infrared wavelengths. "
        "Its observations help scientists study early galaxies, stars, and planetary systems. "
        "The telescope operates near the Sun-Earth L2 point, approximately 1.5 million kilometres from Earth."
    ),
    "international_space_station": (
        "The International Space Station is a habitable research laboratory orbiting Earth. "
        "It has been continuously occupied since November 2000. "
        "Astronauts conduct experiments in biology, physics, medicine, and materials science aboard the station. "
        "Multiple space agencies, including NASA, ESA, JAXA, and Roscosmos, have contributed to its operation."
    ),
    "black_holes": (
        "A black hole is a region of spacetime where gravity is so strong that nothing can escape from inside its event horizon. "
        "Black holes can form when sufficiently massive stars collapse at the end of their lives. "
        "Scientists can study black holes by observing their effects on nearby matter and light. "
        "The Event Horizon Telescope collaboration released the first image of a black hole's shadow in 2019."
    )
}


In [12]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: space_missions]
NASA's Apollo 11 mission landed the first humans on the Moon in July 1969. Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited the Moon. The mission used the Saturn V rocket to travel to space. Apollo 11 returned safely to Earth on July 24, 1969.

[Source: mars_rovers]
NASA's Perseverance rover landed on Mars in February 2021. Its mission includes searching for signs of ancient microbial life. The rover collects and stores rock samples for possible future return to Earth. It also carries an experiment called MOXIE that demonstrated oxygen production from the Martian atmosphere.


In [25]:

# @title Ask Gemini to search for something
import google.generativeai as genai
from google.colab import userdata
import time

genai.configure(api_key=userdata.get("Gemini_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    tools=[search_documents],
    system_instruction=(
        """
        You are an assistant answering questions using a document collection.
        When you need specific information, use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)


Model ready: models/gemini-3.5-flash-lite


In [32]:

import time
from google import genai
from google.genai import types
from google.colab import userdata

# Connect to Gemini using your existing Colab Secret
client = genai.Client(
    api_key=userdata.get("Gemini_API_KEY"),
    http_options=types.HttpOptions(timeout=45000)
)

# Define the search tool Gemini can call
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the document collection for paragraphs relevant to a query.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Keywords to search for in the documents."
                    }
                },
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)

# Send requests and retry temporary server/rate-limit errors
def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents,
                config=config
            )
        except Exception as e:
            error = str(e)
            temporary_error = any(
                code in error for code in ["429", "500", "503", "504"]
            )

            if temporary_error and attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f"Temporary error. Retrying in {wait} seconds...")
                time.sleep(wait)
            else:
                raise

# ReAct agent: reason, call tool, observe result, answer
def run_agent(question: str, max_steps: int = 5) -> str:
    print(f"Question: {question}\n" + "-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)

        if not response.candidates:
            print("Gemini returned no candidates.")
            return "No answer returned by the model."

        model_content = response.candidates[0].content
        contents.append(model_content)

        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        # No tool calls means the model has produced its final answer
        if not calls:
            answer = response.text or "The model returned no text answer."
            print(f"\nFinal answer:\n{answer}")
            return answer

        # Execute requested tools and send their results back to Gemini
        result_parts = []

        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")

            if call.name == "search_documents":
                result = search_documents(**args)
            else:
                result = f"Unknown tool: {call.name}"

            print(f"  Result preview: {str(result)[:150]}...")

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result},
                )
            )

        contents.append(
            types.Content(role="user", parts=result_parts)
        )

    print("\nAgent stopped: reached the step limit.")
    return "Stopped before the model produced a final answer."


# Test the agent
_ = run_agent("What did the Apollo 11 mission accomplish?")


Question: What did the Apollo 11 mission accomplish?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Apollo 11 mission accomplish'})
  Result preview: [Source: space_missions]
NASA's Apollo 11 mission landed the first humans on the Moon in July 1969. Neil Armstrong and Buzz Aldrin walked on the lunar...

Final answer:
The Apollo 11 mission accomplished several historic milestones in July 1969:

* **First Moon Landing:** It successfully landed the first humans on the Moon.
* **Lunar Surface Walk:** Astronauts Neil Armstrong and Buzz Aldrin walked on the lunar surface (with Armstrong famously becoming the first person to step onto the Moon), while their crewmate Michael Collins orbited the Moon.
* **Safe Return:** The mission returned safely to Earth on July 24, 1969, using the Saturn V rocket for space travel.


In [30]:
print(search_documents("Apollo 11"))

[Source: space_missions]
NASA's Apollo 11 mission landed the first humans on the Moon in July 1969. Neil Armstrong and Buzz Aldrin walked on the lunar surface while Michael Collins orbited the Moon. The mission used the Saturn V rocket to travel to space. Apollo 11 returned safely to Earth on July 24, 1969.


In [33]:

# @title Ask follow-up questions

# Questions based on our space documents
questions = [
    "What did the Apollo 11 mission accomplish?",
    "When did the Perseverance rover land on Mars?",
    "What does the James Webb Space Telescope observe?",
    # This question is not answered by the documents
    "How many moons does a fictional planet called Zorbon have?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # Allow time between requests on the free tier

    print("\n" + "=" * 60)
    run_agent(q)



Question: What did the Apollo 11 mission accomplish?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Apollo 11 mission accomplish'})
  Result preview: [Source: space_missions]
NASA's Apollo 11 mission landed the first humans on the Moon in July 1969. Neil Armstrong and Buzz Aldrin walked on the lunar...

Final answer:
The Apollo 11 mission accomplished several historic milestones in July 1969:

* **First Moon Landing:** It landed the first humans on the Moon.
* **Lunar Surface Walk:** Astronauts Neil Armstrong and Buzz Aldrin walked on the lunar surface (with Armstrong famously becoming the first person to step onto the Moon), while their crewmate Michael Collins orbited the Moon.
* **Safe Return:** The mission successfully returned all three astronauts safely to Earth on July 24, 1969, using the Saturn V rocket.

Question: When did the Perseverance rover land on Mars?
--------------------------------------------------
Step 1: model c

In [34]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)